In [1]:
# ======================
# CONFIG
# ======================
import torch
import json
import csv
import re
import unicodedata
import difflib
from typing import List, Set, Tuple, Dict
from collections import defaultdict
from transformers import (
    MBart50TokenizerFast,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    MBart50TokenizerFast

)
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# MODEL_PATH      = r"C:\Users\Lenovo\Desktop\Nepali_GEC\nepali_gec\outputs\best_model_v2s_v4_FF"
# MODEL_PATH      = r"tuyal/Stage2FFTmBart"
MODEL_PATH      = r"tuyal/FFTmBart_BM"
TEST_JSON       = r"C:\Users\Lenovo\Desktop\Nepali_GEC\nepali_gec\tests\test.json"
OUTPUT_CSV_PATH = r"C:\Users\Lenovo\Desktop\Nepali_GEC\nepali_gec\tests\predictions.csv"
OUTPUT_TXT_PATH = r"C:\Users\Lenovo\Desktop\Nepali_GEC\nepali_gec\tests\predictions.txt"

MAX_LENGTH     = 64
NUM_BEAMS      = 5
NUM_RETURN_SEQ = 5   # number of beam candidates kept
# PREFIX         = "Correct sentence: "
PREFIX         = ""

# Set to None to skip reranking
RERANKER_MODEL = "AIsumit123/muril-reranker-mbart"

# ======================
# NORMALIZATION
# ======================

def normalize(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("|", "।")
    text = re.sub(r"\s*।\s*", "।", text)
    text = re.sub(r"।+", "।", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def tokenize(text: str) -> List[str]:
    text = normalize(text)
    return re.findall(r'[\u0900-\u097F]+|[^\u0900-\u097F\s]+|\S', text)

# ======================
# EDIT EXTRACTION & METRICS
# ======================

def extract_edits(source: str, target: str) -> Set[Tuple[str, str]]:
    src_tok = tokenize(source)
    tgt_tok = tokenize(target)
    matcher = difflib.SequenceMatcher(None, src_tok, tgt_tok, autojunk=False)
    edits = set()
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            continue
        edits.add((" ".join(src_tok[i1:i2]), " ".join(tgt_tok[j1:j2])))
    return edits

def compute_f05(tp: float, fp: float, fn: float) -> Tuple[float, float, float]:
    p   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r   = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    b2  = 0.25  # beta=0.5 squared
    f05 = ((1 + b2) * p * r / (b2 * p + r)) if (p + r) > 0 else 0.0
    return p, r, f05

# ======================
# GLEU (sentence-level)
# ======================

def _ngrams(tokens: List[str], n: int) -> Dict[tuple, int]:
    counts: Dict[tuple, int] = {}
    for i in range(len(tokens) - n + 1):
        ng = tuple(tokens[i:i+n])
        counts[ng] = counts.get(ng, 0) + 1
    return counts

def gleu_sentence(source: str, hyp: str, ref: str, max_n: int = 4) -> float:
    src_tok = tokenize(source)
    hyp_tok = tokenize(hyp)
    ref_tok = tokenize(ref)
    if not hyp_tok:
        return 0.0
    total_match = total_hyp = total_ref = 0
    for n in range(1, max_n + 1):
        hyp_ng = _ngrams(hyp_tok, n)
        ref_ng = _ngrams(ref_tok, n)
        src_ng = _ngrams(src_tok, n)
        for ng, cnt in hyp_ng.items():
            match     = min(cnt, ref_ng.get(ng, 0))
            src_match = min(cnt, src_ng.get(ng, 0)) if ng in src_ng and ng not in ref_ng else 0
            total_match += max(0, match - src_match)
        total_hyp += max(0, len(hyp_tok) - n + 1)
        total_ref += max(0, len(ref_tok) - n + 1)
    if total_hyp == 0 or total_ref == 0:
        return 0.0
    prec = total_match / total_hyp
    rec  = total_match / total_ref
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

# ======================
# REFERENCE HELPERS
# ======================

def get_references(item: dict) -> List[str]:
    if "references" in item:
        return item["references"]
    if "correct_sentence" in item:
        return [item["correct_sentence"]]
    return []

# ======================
# SCORING HELPERS
# ======================

def score_sentence(
    source: str, prediction: str, references: List[str]
) -> Tuple[float, float, float, str]:
    """Oracle multi-reference F0.5 — picks the ref that maximises F0.5."""
    sys_edits = extract_edits(source, prediction)
    best_f05  = -1.0
    best_tp = best_fp = best_fn = 0.0
    best_ref = references[0] if references else ""

    for ref in references:
        gold = extract_edits(source, ref)
        tp   = float(len(sys_edits & gold))
        fp   = float(len(sys_edits - gold))
        fn   = float(len(gold - sys_edits))
        # Special case: noop sentence AND noop prediction → perfect score
        if len(gold) == 0 and len(sys_edits) == 0:
            f05 = 1.0
            tp = fp = fn = 0.0
        else:
            _, _, f05 = compute_f05(tp, fp, fn)
        if f05 > best_f05:
            best_f05 = f05
            best_tp, best_fp, best_fn = tp, fp, fn
            best_ref = ref
    return best_tp, best_fp, best_fn, best_ref

def oracle_score(
    source: str, candidates: List[str], references: List[str]
) -> Tuple[float, float, float]:
    """Return TP/FP/FN of the best candidate in the list (oracle selection)."""
    best_f05 = -1.0
    best_tp = best_fp = best_fn = 0.0
    for cand in candidates:
        tp, fp, fn, _ = score_sentence(source, cand, references)
        _, _, f05 = compute_f05(tp, fp, fn)
        if f05 > best_f05:
            best_f05 = f05
            best_tp, best_fp, best_fn = tp, fp, fn
    return best_tp, best_fp, best_fn

# ======================
# LOAD MODELS
# ======================

print(f"Loading generator from {MODEL_PATH} ...")
gen_tok   = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50")
# gen_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH, device_map=DEVICE)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,   # reduces VRAM, avoids some OOM-triggered asserts
).to(DEVICE)
gen_tok.src_lang = "ne_NP"
gen_tok.tgt_lang = "ne_NP"
gen_model.eval()
print("Generator loaded ✅")
from transformers import AutoTokenizer
rerank_tok = rerank_model = None
SEP = "[SEP]"
if RERANKER_MODEL:
    print(f"Loading reranker ({RERANKER_MODEL}) ...")
    # rerank_tok   = MBart50TokenizerFast.from_pretrained(RERANKER_MODEL, trust_remote_code=True)
    
    rerank_tok = AutoTokenizer.from_pretrained(RERANKER_MODEL, trust_remote_code=True)
    rerank_model = AutoModelForSequenceClassification.from_pretrained(
        RERANKER_MODEL, trust_remote_code=True).to(DEVICE)
    rerank_model.eval()
    # FIX: MuRIL and some other tokenizers return sep_token=None
    SEP = rerank_tok.sep_token or "[SEP]"
    print(f"Reranker loaded ✅  (sep token = {repr(SEP)})")
else:
    print("⚠️  Reranker disabled — using generator top-1 only.")

# ======================
# GENERATION
# ======================

def generate_candidates(input_text: str) -> List[str]:
    prefixed = PREFIX + input_text
    enc = gen_tok(
        prefixed, return_tensors="pt",
        truncation=True, max_length=MAX_LENGTH
    ).to(DEVICE)
    with torch.no_grad():
        out = gen_model.generate(
            **enc,
            num_beams=NUM_BEAMS,
            num_return_sequences=NUM_RETURN_SEQ,
            max_length=MAX_LENGTH,
            early_stopping=True,
            forced_bos_token_id=gen_tok.lang_code_to_id["ne_NP"]
        )
    cands = gen_tok.batch_decode(out, skip_special_tokens=True)
    cands = [c.strip() for c in cands]
    return list(dict.fromkeys(cands))   # deduplicate, preserve beam order

# ======================
# PAIRWISE RERANKING
# ======================

def compare_pair(source: str, cand_A: str, cand_B: str) -> float:
    """Returns P(A is better than B)."""
    enc = rerank_tok(
        source,
        # f"{cand_A} {SEP} {cand_B}",
        cand_A + " " + cand_B,
        return_tensors="pt", truncation=True, max_length=128
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        probs = torch.softmax(rerank_model(**enc).logits, dim=-1)
    return probs[0][1].item()

def tournament_rerank(source: str, candidates: List[str]) -> List[str]:
    """Tournament-style pairwise reranking. Returns candidates sorted best-first."""
    n    = len(candidates)
    wins = [0] * n
    for i in range(n):
        for j in range(i + 1, n):
            if compare_pair(source, candidates[i], candidates[j]) > 0.5:
                wins[i] += 1
            else:
                wins[j] += 1
    ranked = sorted(range(n), key=lambda x: wins[x], reverse=True)
    return [candidates[i] for i in ranked]

# ======================
# LOAD TEST DATA
# ======================

with open(TEST_JSON, "r", encoding="utf-8") as f:
    test_data = json.load(f)
print(f"Loaded {len(test_data)} test samples ✅")
print(f"Sample keys: {list(test_data[0].keys())}")

# ======================
# GENERATE + RERANK
# ======================

print("\nGenerating predictions...")
for idx, item in enumerate(test_data):
    src = item["incorrect_sentence"]

    # Raw beam-search candidates (ordered best→worst by beam score)
    raw_cands = generate_candidates(src)
    item["raw_candidates"] = raw_cands
    item["pred_no_rerank"] = raw_cands[0] if raw_cands else src   # top-1 beam

    # Reranked candidates
    if rerank_model is not None:
        ranked_cands = tournament_rerank(src, raw_cands)
    else:
        ranked_cands = raw_cands   # no reranking → same order

    item["ranked_candidates"] = ranked_cands
    item["pred_reranked"]     = ranked_cands[0] if ranked_cands else src

    if (idx + 1) % 10 == 0 or (idx + 1) == len(test_data):
        print(f"  {idx + 1}/{len(test_data)} done")

print("✅ Generation complete\n")

# ======================
# UNIFIED EVALUATION LOOP
# Scores BOTH predictions in one pass to avoid duplicate oracle calls
# ======================

# Accumulators for no-rerank (nr) and reranked (rr)
def make_agg():
    return dict(
        tp=0., fp=0., fn=0.,
        oracle_tp=0., oracle_fp=0., oracle_fn=0.,
        oracle5_tp=0., oracle5_fp=0., oracle5_fn=0.,
        exact=0, gleu=[], noop_correct=0, noop_total=0,
        recall={k: 0 for k in range(1, NUM_RETURN_SEQ + 1)},
        type_stats=defaultdict(lambda: dict(tp=0., fp=0., fn=0.)),
    )

agg = {"nr": make_agg(), "rr": make_agg()}

for item in test_data:
    src   = item["incorrect_sentence"]
    refs  = get_references(item)
    etype = item.get("error_type", "UNKNOWN")

    if not refs:
        print(f"⚠️  No reference for: {src[:50]}")
        continue

    raw_cands    = item["raw_candidates"]
    ranked_cands = item["ranked_candidates"]

    # Score each prediction variant
    # FIX: no-rerank oracle uses raw_cands; reranked oracle uses ranked_cands
    for key, pred, cands_for_oracle in (
        ("nr", item["pred_no_rerank"], raw_cands),
        ("rr", item["pred_reranked"],  ranked_cands),
    ):
        a = agg[key]

        # F0.5 (oracle multi-reference)
        tp, fp, fn, best_ref = score_sentence(src, pred, refs)
        a["tp"] += tp; a["fp"] += fp; a["fn"] += fn
        a["type_stats"][etype]["tp"] += tp
        a["type_stats"][etype]["fp"] += fp
        a["type_stats"][etype]["fn"] += fn

        # Store per-item for CSV
        item[f"_tp_{key}"]       = tp
        item[f"_fp_{key}"]       = fp
        item[f"_fn_{key}"]       = fn
        item[f"_best_ref_{key}"] = best_ref

        # Exact match (any reference)
        if any(normalize(pred) == normalize(r) for r in refs):
            a["exact"] += 1

        # GLEU
        a["gleu"].append(gleu_sentence(src, pred, best_ref))

        # Oracle@K
        o_tp,  o_fp,  o_fn  = oracle_score(src, cands_for_oracle[:NUM_RETURN_SEQ], refs)
        o5_tp, o5_fp, o5_fn = oracle_score(src, cands_for_oracle[:5], refs)
        a["oracle_tp"]  += o_tp;  a["oracle_fp"]  += o_fp;  a["oracle_fn"]  += o_fn
        a["oracle5_tp"] += o5_tp; a["oracle5_fp"] += o5_fp; a["oracle5_fn"] += o5_fn

        # Recall@K
        for k in a["recall"]:
            top_k = cands_for_oracle[:k]
            if any(normalize(c) == normalize(r) for c in top_k for r in refs):
                a["recall"][k] += 1

        # Noop accuracy
        gold_primary = extract_edits(src, refs[0])
        if len(gold_primary) == 0:
            a["noop_total"] += 1
            if normalize(pred) == normalize(refs[0]):
                a["noop_correct"] += 1

# ======================
# PRINT RESULTS
# ======================

LABELS = {"nr": "MT5 top-1  (no reranker)", "rr": "MT5 + reranker"}
PRED_KEYS = {"nr": "pred_no_rerank", "rr": "pred_reranked"}

def print_results_for(key: str) -> dict:
    a = agg[key]
    n = len(test_data)
    p,   r,   f05   = compute_f05(a["tp"],        a["fp"],        a["fn"])
    o5p, o5r, o5f05 = compute_f05(a["oracle5_tp"], a["oracle5_fp"], a["oracle5_fn"])
    op,  or_,  of05  = compute_f05(a["oracle_tp"],  a["oracle_fp"],  a["oracle_fn"])
    em       = a["exact"] / n if n > 0 else 0.0
    avg_gleu = sum(a["gleu"]) / len(a["gleu"]) if a["gleu"] else 0.0
    noop_acc = a["noop_correct"] / a["noop_total"] if a["noop_total"] > 0 else 0.0

    sep = "=" * 58
    print(f"\n{sep}")
    print(f"   RESULTS  —  {LABELS[key]}")
    print(sep)
    print(f"  Exact Match Rate    : {em:.4f}  ({a['exact']}/{n})")
    print(f"  Precision  (F0.5)   : {p:.4f}")
    print(f"  Recall     (F0.5)   : {r:.4f}")
    print(f"  F0.5                : {f05:.4f}")
    print(f"  GLEU                : {avg_gleu:.4f}")
    print(f"  Oracle F0.5@5       : {o5f05:.4f}  (P={o5p:.4f}  R={o5r:.4f})")
    for k in sorted(a["recall"]):
        print(f"  Recall@{k:<2}            : {a['recall'][k]/n:.4f}  ({a['recall'][k]}/{n})")
    print(f"  Noop Accuracy       : {noop_acc:.4f}  ({a['noop_correct']}/{a['noop_total']})")
    print(f"  TP / FP / FN        : {a['tp']:.1f} / {a['fp']:.1f} / {a['fn']:.1f}")

    pred_key_name = PRED_KEYS[key]
    print(f"\n{'─'*58}")
    print(f"  PER ERROR TYPE  —  {LABELS[key]}")
    print(f"{'─'*58}")
    for etype, stats in sorted(a["type_stats"].items()):
        pe, re_, f05e = compute_f05(stats["tp"], stats["fp"], stats["fn"])
        total_e = sum(1 for it in test_data if it.get("error_type", "UNKNOWN") == etype)
        em_e    = sum(
            1 for it in test_data
            if it.get("error_type", "UNKNOWN") == etype
            and any(normalize(it[pred_key_name]) == normalize(ref)
                    for ref in get_references(it))
        )
        print(f"\n  [{etype}]  ({total_e} samples,  EM={em_e}/{total_e})")
        print(f"    P={pe:.4f}  R={re_:.4f}  F0.5={f05e:.4f}")
        print(f"    TP={stats['tp']:.1f}  FP={stats['fp']:.1f}  FN={stats['fn']:.1f}")

    return {"em": em, "p": p, "r": r, "f05": f05, "gleu": avg_gleu,
            "oracle5_f05": o5f05, "oracle_f05": of05, "noop_acc": noop_acc}

res_nr = print_results_for("nr")
res_rr = print_results_for("rr")

# ======================
# SIDE-BY-SIDE COMPARISON
# ======================

print("\n" + "=" * 60)
print("   SIDE-BY-SIDE COMPARISON")
print("=" * 60)
metrics = [
    ("Exact Match",          "em"),
    ("Precision",            "p"),
    ("Recall",               "r"),
    ("F0.5",                 "f05"),
    ("GLEU",                 "gleu"),
    ("Oracle F0.5@5",        "oracle5_f05"),
    ("Noop Accuracy",        "noop_acc"),
]
print(f"  {'Metric':<24}  {'No Reranker':>11}  {'Reranked':>11}  {'Δ':>8}")
print(f"  {'─'*24}  {'─'*11}  {'─'*11}  {'─'*8}")
for name, k in metrics:
    v1, v2 = res_nr[k], res_rr[k]
    delta  = v2 - v1
    sign   = "+" if delta >= 0 else ""
    print(f"  {name:<24}  {v1:>11.4f}  {v2:>11.4f}  {sign}{delta:.4f}")

# ======================
# SAVE CSV
# ======================

print(f"\nSaving CSV → {OUTPUT_CSV_PATH}")

raw_headers    = [f"raw_beam_{k}"      for k in range(1, NUM_RETURN_SEQ + 1)]
ranked_headers = [f"reranked_cand_{k}" for k in range(1, NUM_RETURN_SEQ + 1)]
header = [
    "#", "source",
    "pred_no_rerank",  "best_ref_nr", "match_nr", "TP_nr", "FP_nr", "FN_nr",
    "pred_reranked",   "best_ref_rr", "match_rr", "TP_rr", "FP_rr", "FN_rr",
    "error_type",
] + raw_headers + ranked_headers

with open(OUTPUT_CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    for idx, item in enumerate(test_data, 1):
        src   = item["incorrect_sentence"]
        refs  = get_references(item)
        etype = item.get("error_type", "UNKNOWN")

        pred_nr = item["pred_no_rerank"]
        pred_rr = item["pred_reranked"]

        match_nr = any(normalize(pred_nr) == normalize(r) for r in refs)
        match_rr = any(normalize(pred_rr) == normalize(r) for r in refs)

        raw_cands    = item.get("raw_candidates",    [])
        ranked_cands = item.get("ranked_candidates", [])
        raw_vals    = (raw_cands    + [""] * NUM_RETURN_SEQ)[:NUM_RETURN_SEQ]
        ranked_vals = (ranked_cands + [""] * NUM_RETURN_SEQ)[:NUM_RETURN_SEQ]

        writer.writerow([
            idx, src,
            pred_nr, item.get("_best_ref_nr", ""), match_nr,
            round(item.get("_tp_nr", 0), 2),
            round(item.get("_fp_nr", 0), 2),
            round(item.get("_fn_nr", 0), 2),
            pred_rr, item.get("_best_ref_rr", ""), match_rr,
            round(item.get("_tp_rr", 0), 2),
            round(item.get("_fp_rr", 0), 2),
            round(item.get("_fn_rr", 0), 2),
            etype,
            *raw_vals,
            *ranked_vals,
        ])

print("✅ CSV saved")

# ======================
# SAVE TXT (human-readable, one sample per block)
# ======================

print(f"\nSaving TXT  → {OUTPUT_TXT_PATH}")

SEP_MAJOR = "=" * 60
SEP_MINOR = "-" * 60

with open(OUTPUT_TXT_PATH, "w", encoding="utf-8") as f:
    for idx, item in enumerate(test_data, 1):
        src   = item["incorrect_sentence"]
        refs  = get_references(item)
        etype = item.get("error_type", "UNKNOWN")

        pred_nr = item["pred_no_rerank"]
        pred_rr = item["pred_reranked"]

        match_nr = any(normalize(pred_nr) == normalize(r) for r in refs)
        match_rr = any(normalize(pred_rr) == normalize(r) for r in refs)

        tp_nr = round(item.get("_tp_nr", 0), 2)
        fp_nr = round(item.get("_fp_nr", 0), 2)
        fn_nr = round(item.get("_fn_nr", 0), 2)
        tp_rr = round(item.get("_tp_rr", 0), 2)
        fp_rr = round(item.get("_fp_rr", 0), 2)
        fn_rr = round(item.get("_fn_rr", 0), 2)

        raw_cands    = item.get("raw_candidates",    [])
        ranked_cands = item.get("ranked_candidates", [])

        f.write(f"{SEP_MAJOR}\n")
        f.write(f"Sample #{idx:<4}  |  {etype}\n")
        f.write(f"{SEP_MAJOR}\n")
        f.write(f"Source        : {src}\n")
        for i, ref in enumerate(refs, 1):
            label = "Reference" if len(refs) == 1 else f"Reference {i}"
            f.write(f"{label:<14}: {ref}\n")

        f.write(f"\n{SEP_MINOR}\n")
        f.write(f"No reranker\n")
        f.write(f"{SEP_MINOR}\n")
        f.write(f"Prediction    : {pred_nr}\n")
        f.write(f"Match         : {match_nr}   TP={tp_nr}  FP={fp_nr}  FN={fn_nr}\n")

        f.write(f"\n{SEP_MINOR}\n")
        f.write(f"Reranked\n")
        f.write(f"{SEP_MINOR}\n")
        f.write(f"Prediction    : {pred_rr}\n")
        f.write(f"Match         : {match_rr}   TP={tp_rr}  FP={fp_rr}  FN={fn_rr}\n")

        f.write(f"\n{SEP_MINOR}\n")
        f.write(f"Raw beam candidates (generator order)\n")
        f.write(f"{SEP_MINOR}\n")
        for i, cand in enumerate(raw_cands, 1):
            f.write(f"  {i}. {cand}\n")

        f.write(f"\n{SEP_MINOR}\n")
        f.write(f"Reranked candidates\n")
        f.write(f"{SEP_MINOR}\n")
        for i, cand in enumerate(ranked_cands, 1):
            f.write(f"  {i}. {cand}\n")

        f.write("\n\n")

print("✅ TXT saved")

# ======================
# SAMPLE PREDICTIONS (sanity check, prints first 5 to console)
# ======================

print("\n===== SAMPLE PREDICTIONS (first 5) =====")
for item in test_data[:5]:
    src  = item["incorrect_sentence"]
    refs = get_references(item)
    nr   = item["pred_no_rerank"]
    rr   = item["pred_reranked"]
    print(f"\nSRC          : {src}")
    print(f"PRED (no rr) : {nr}  match={any(normalize(nr)==normalize(r) for r in refs)}")
    print(f"PRED (rrnk)  : {rr}  match={any(normalize(rr)==normalize(r) for r in refs)}")
    print(f"REF          : {refs[0]}")
    print(f"sys_edits    : {extract_edits(src, nr)}")
    print(f"gold_edits   : {extract_edits(src, refs[0])}")

c:\Users\Lenovo\Desktop\Nepali_GEC\nepali_gec\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading generator from tuyal/FFTmBart_BM ...


`torch_dtype` is deprecated! Use `dtype` instead!
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Generator loaded ✅
Loading reranker (AIsumit123/muril-reranker-mbart) ...
Reranker loaded ✅  (sep token = '[SEP]')
Loaded 414 test samples ✅
Sample keys: ['incorrect_sentence', 'correct_sentence', 'error_type', 'edit']

Generating predictions...
  10/414 done
  20/414 done
  30/414 done
  40/414 done
  50/414 done
  60/414 done
  70/414 done
  80/414 done
  90/414 done
  100/414 done
  110/414 done
  120/414 done
  130/414 done
  140/414 done
  150/414 done
  160/414 done
  170/414 done
  180/414 done
  190/414 done
  200/414 done
  210/414 done
  220/414 done
  230/414 done
  240/414 done
  250/414 done
  260/414 done
  270/414 done
  280/414 done
  290/414 done
  300/414 done
  310/414 done
  320/414 done
  330/414 done
  340/414 done
  350/414 done
  360/414 done
  370/414 done
  380/414 done
  390/414 done
  400/414 done
  410/414 done
  414/414 done
✅ Generation complete


   RESULTS  —  MT5 top-1  (no reranker)
  Exact Match Rate    : 0.2947  (122/414)
  Precision  (F0.5)   : 0.4

In [4]:
# ======================
# CONFIG
# ======================
import torch
import json
import csv
import re
import unicodedata
import difflib
from typing import List, Set, Tuple, Dict
from collections import defaultdict
from transformers import (
    MBart50TokenizerFast,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    MBart50TokenizerFast

)
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# MODEL_PATH      = r"C:\Users\Lenovo\Desktop\Nepali_GEC\nepali_gec\outputs\best_model_v2s_v4_FF"
MODEL_PATH      = r"tuyal/Stage2FFTmBart"
TEST_JSON       = r"C:\Users\Lenovo\Desktop\Nepali_GEC\nepali_gec\tests\test.json"
OUTPUT_CSV_PATH = r"C:\Users\Lenovo\Desktop\Nepali_GEC\nepali_gec\tests\predictions.csv"
OUTPUT_TXT_PATH = r"C:\Users\Lenovo\Desktop\Nepali_GEC\nepali_gec\tests\predictions.txt"

MAX_LENGTH     = 64
NUM_BEAMS      = 5
NUM_RETURN_SEQ = 5   # number of beam candidates kept
# PREFIX         = "Correct sentence: "
PREFIX         = ""

# Set to None to skip reranking
RERANKER_MODEL = "AIsumit123/muril-reranker-mbartF"

# ======================
# NORMALIZATION
# ======================

def normalize(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("|", "।")
    text = re.sub(r"\s*।\s*", "।", text)
    text = re.sub(r"।+", "।", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def tokenize(text: str) -> List[str]:
    text = normalize(text)
    return re.findall(r'[\u0900-\u097F]+|[^\u0900-\u097F\s]+|\S', text)

# ======================
# EDIT EXTRACTION & METRICS
# ======================

def extract_edits(source: str, target: str) -> Set[Tuple[str, str]]:
    src_tok = tokenize(source)
    tgt_tok = tokenize(target)
    matcher = difflib.SequenceMatcher(None, src_tok, tgt_tok, autojunk=False)
    edits = set()
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            continue
        edits.add((" ".join(src_tok[i1:i2]), " ".join(tgt_tok[j1:j2])))
    return edits

def compute_f05(tp: float, fp: float, fn: float) -> Tuple[float, float, float]:
    p   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r   = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    b2  = 0.25  # beta=0.5 squared
    f05 = ((1 + b2) * p * r / (b2 * p + r)) if (p + r) > 0 else 0.0
    return p, r, f05

# ======================
# GLEU (sentence-level)
# ======================

def _ngrams(tokens: List[str], n: int) -> Dict[tuple, int]:
    counts: Dict[tuple, int] = {}
    for i in range(len(tokens) - n + 1):
        ng = tuple(tokens[i:i+n])
        counts[ng] = counts.get(ng, 0) + 1
    return counts

def gleu_sentence(source: str, hyp: str, ref: str, max_n: int = 4) -> float:
    src_tok = tokenize(source)
    hyp_tok = tokenize(hyp)
    ref_tok = tokenize(ref)
    if not hyp_tok:
        return 0.0
    total_match = total_hyp = total_ref = 0
    for n in range(1, max_n + 1):
        hyp_ng = _ngrams(hyp_tok, n)
        ref_ng = _ngrams(ref_tok, n)
        src_ng = _ngrams(src_tok, n)
        for ng, cnt in hyp_ng.items():
            match     = min(cnt, ref_ng.get(ng, 0))
            src_match = min(cnt, src_ng.get(ng, 0)) if ng in src_ng and ng not in ref_ng else 0
            total_match += max(0, match - src_match)
        total_hyp += max(0, len(hyp_tok) - n + 1)
        total_ref += max(0, len(ref_tok) - n + 1)
    if total_hyp == 0 or total_ref == 0:
        return 0.0
    prec = total_match / total_hyp
    rec  = total_match / total_ref
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

# ======================
# REFERENCE HELPERS
# ======================

def get_references(item: dict) -> List[str]:
    if "references" in item:
        return item["references"]
    if "correct_sentence" in item:
        return [item["correct_sentence"]]
    return []

# ======================
# SCORING HELPERS
# ======================

def score_sentence(
    source: str, prediction: str, references: List[str]
) -> Tuple[float, float, float, str]:
    """Oracle multi-reference F0.5 — picks the ref that maximises F0.5."""
    sys_edits = extract_edits(source, prediction)
    best_f05  = -1.0
    best_tp = best_fp = best_fn = 0.0
    best_ref = references[0] if references else ""

    for ref in references:
        gold = extract_edits(source, ref)
        tp   = float(len(sys_edits & gold))
        fp   = float(len(sys_edits - gold))
        fn   = float(len(gold - sys_edits))
        # Special case: noop sentence AND noop prediction → perfect score
        if len(gold) == 0 and len(sys_edits) == 0:
            f05 = 1.0
            tp = fp = fn = 0.0
        else:
            _, _, f05 = compute_f05(tp, fp, fn)
        if f05 > best_f05:
            best_f05 = f05
            best_tp, best_fp, best_fn = tp, fp, fn
            best_ref = ref
    return best_tp, best_fp, best_fn, best_ref

def oracle_score(
    source: str, candidates: List[str], references: List[str]
) -> Tuple[float, float, float]:
    """Return TP/FP/FN of the best candidate in the list (oracle selection)."""
    best_f05 = -1.0
    best_tp = best_fp = best_fn = 0.0
    for cand in candidates:
        tp, fp, fn, _ = score_sentence(source, cand, references)
        _, _, f05 = compute_f05(tp, fp, fn)
        if f05 > best_f05:
            best_f05 = f05
            best_tp, best_fp, best_fn = tp, fp, fn
    return best_tp, best_fp, best_fn

# ======================
# LOAD MODELS
# ======================

print(f"Loading generator from {MODEL_PATH} ...")
gen_tok   = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50")
# gen_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH, device_map=DEVICE)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,   # reduces VRAM, avoids some OOM-triggered asserts
).to(DEVICE)
gen_tok.src_lang = "ne_NP"
gen_tok.tgt_lang = "ne_NP"
gen_model.eval()
print("Generator loaded ✅")
from transformers import AutoTokenizer
rerank_tok = rerank_model = None
SEP = "[SEP]"
if RERANKER_MODEL:
    print(f"Loading reranker ({RERANKER_MODEL}) ...")
    # rerank_tok   = MBart50TokenizerFast.from_pretrained(RERANKER_MODEL, trust_remote_code=True)
    
    rerank_tok = AutoTokenizer.from_pretrained(RERANKER_MODEL, trust_remote_code=True)
    rerank_model = AutoModelForSequenceClassification.from_pretrained(
        RERANKER_MODEL, trust_remote_code=True).to(DEVICE)
    rerank_model.eval()
    # FIX: MuRIL and some other tokenizers return sep_token=None
    SEP = rerank_tok.sep_token or "[SEP]"
    print(f"Reranker loaded ✅  (sep token = {repr(SEP)})")
else:
    print("⚠️  Reranker disabled — using generator top-1 only.")

# ======================
# GENERATION
# ======================

def generate_candidates(input_text: str) -> List[str]:
    prefixed = PREFIX + input_text
    enc = gen_tok(
        prefixed, return_tensors="pt",
        truncation=True, max_length=MAX_LENGTH
    ).to(DEVICE)
    with torch.no_grad():
        out = gen_model.generate(
            **enc,
            num_beams=NUM_BEAMS,
            num_return_sequences=NUM_RETURN_SEQ,
            max_length=MAX_LENGTH,
            early_stopping=True,
            forced_bos_token_id=gen_tok.lang_code_to_id["ne_NP"]
        )
    cands = gen_tok.batch_decode(out, skip_special_tokens=True)
    cands = [c.strip() for c in cands]
    return list(dict.fromkeys(cands))   # deduplicate, preserve beam order

# ======================
# PAIRWISE RERANKING
# ======================

def compare_pair(source: str, cand_A: str, cand_B: str) -> float:
    """Returns P(A is better than B)."""
    enc = rerank_tok(
        source,
        # f"{cand_A} {SEP} {cand_B}",
        cand_A + " " + cand_B,
        return_tensors="pt", truncation=True, max_length=128
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        probs = torch.softmax(rerank_model(**enc).logits, dim=-1)
    return probs[0][1].item()

def tournament_rerank(source: str, candidates: List[str]) -> List[str]:
    """Tournament-style pairwise reranking. Returns candidates sorted best-first."""
    n    = len(candidates)
    wins = [0] * n
    for i in range(n):
        for j in range(i + 1, n):
            if compare_pair(source, candidates[i], candidates[j]) > 0.5:
                wins[i] += 1
            else:
                wins[j] += 1
    ranked = sorted(range(n), key=lambda x: wins[x], reverse=True)
    return [candidates[i] for i in ranked]

# ======================
# LOAD TEST DATA
# ======================

with open(TEST_JSON, "r", encoding="utf-8") as f:
    test_data = json.load(f)
print(f"Loaded {len(test_data)} test samples ✅")
print(f"Sample keys: {list(test_data[0].keys())}")

# ======================
# GENERATE + RERANK
# ======================

print("\nGenerating predictions...")
for idx, item in enumerate(test_data):
    src = item["incorrect_sentence"]

    # Raw beam-search candidates (ordered best→worst by beam score)
    raw_cands = generate_candidates(src)
    item["raw_candidates"] = raw_cands
    item["pred_no_rerank"] = raw_cands[0] if raw_cands else src   # top-1 beam

    # Reranked candidates
    if rerank_model is not None:
        ranked_cands = tournament_rerank(src, raw_cands)
    else:
        ranked_cands = raw_cands   # no reranking → same order

    item["ranked_candidates"] = ranked_cands
    item["pred_reranked"]     = ranked_cands[0] if ranked_cands else src

    if (idx + 1) % 10 == 0 or (idx + 1) == len(test_data):
        print(f"  {idx + 1}/{len(test_data)} done")

print("✅ Generation complete\n")

# ======================
# UNIFIED EVALUATION LOOP
# Scores BOTH predictions in one pass to avoid duplicate oracle calls
# ======================

# Accumulators for no-rerank (nr) and reranked (rr)
def make_agg():
    return dict(
        tp=0., fp=0., fn=0.,
        oracle_tp=0., oracle_fp=0., oracle_fn=0.,
        oracle5_tp=0., oracle5_fp=0., oracle5_fn=0.,
        exact=0, gleu=[], noop_correct=0, noop_total=0,
        recall={k: 0 for k in range(1, NUM_RETURN_SEQ + 1)},
        type_stats=defaultdict(lambda: dict(tp=0., fp=0., fn=0.)),
    )

agg = {"nr": make_agg(), "rr": make_agg()}

for item in test_data:
    src   = item["incorrect_sentence"]
    refs  = get_references(item)
    etype = item.get("error_type", "UNKNOWN")

    if not refs:
        print(f"⚠️  No reference for: {src[:50]}")
        continue

    raw_cands    = item["raw_candidates"]
    ranked_cands = item["ranked_candidates"]

    # Score each prediction variant
    # FIX: no-rerank oracle uses raw_cands; reranked oracle uses ranked_cands
    for key, pred, cands_for_oracle in (
        ("nr", item["pred_no_rerank"], raw_cands),
        ("rr", item["pred_reranked"],  ranked_cands),
    ):
        a = agg[key]

        # F0.5 (oracle multi-reference)
        tp, fp, fn, best_ref = score_sentence(src, pred, refs)
        a["tp"] += tp; a["fp"] += fp; a["fn"] += fn
        a["type_stats"][etype]["tp"] += tp
        a["type_stats"][etype]["fp"] += fp
        a["type_stats"][etype]["fn"] += fn

        # Store per-item for CSV
        item[f"_tp_{key}"]       = tp
        item[f"_fp_{key}"]       = fp
        item[f"_fn_{key}"]       = fn
        item[f"_best_ref_{key}"] = best_ref

        # Exact match (any reference)
        if any(normalize(pred) == normalize(r) for r in refs):
            a["exact"] += 1

        # GLEU
        a["gleu"].append(gleu_sentence(src, pred, best_ref))

        # Oracle@K
        o_tp,  o_fp,  o_fn  = oracle_score(src, cands_for_oracle[:NUM_RETURN_SEQ], refs)
        o5_tp, o5_fp, o5_fn = oracle_score(src, cands_for_oracle[:5], refs)
        a["oracle_tp"]  += o_tp;  a["oracle_fp"]  += o_fp;  a["oracle_fn"]  += o_fn
        a["oracle5_tp"] += o5_tp; a["oracle5_fp"] += o5_fp; a["oracle5_fn"] += o5_fn

        # Recall@K
        for k in a["recall"]:
            top_k = cands_for_oracle[:k]
            if any(normalize(c) == normalize(r) for c in top_k for r in refs):
                a["recall"][k] += 1

        # Noop accuracy
        gold_primary = extract_edits(src, refs[0])
        if len(gold_primary) == 0:
            a["noop_total"] += 1
            if normalize(pred) == normalize(refs[0]):
                a["noop_correct"] += 1

# ======================
# PRINT RESULTS
# ======================

LABELS = {"nr": "MT5 top-1  (no reranker)", "rr": "MT5 + reranker"}
PRED_KEYS = {"nr": "pred_no_rerank", "rr": "pred_reranked"}

def print_results_for(key: str) -> dict:
    a = agg[key]
    n = len(test_data)
    p,   r,   f05   = compute_f05(a["tp"],        a["fp"],        a["fn"])
    o5p, o5r, o5f05 = compute_f05(a["oracle5_tp"], a["oracle5_fp"], a["oracle5_fn"])
    op,  or_,  of05  = compute_f05(a["oracle_tp"],  a["oracle_fp"],  a["oracle_fn"])
    em       = a["exact"] / n if n > 0 else 0.0
    avg_gleu = sum(a["gleu"]) / len(a["gleu"]) if a["gleu"] else 0.0
    noop_acc = a["noop_correct"] / a["noop_total"] if a["noop_total"] > 0 else 0.0

    sep = "=" * 58
    print(f"\n{sep}")
    print(f"   RESULTS  —  {LABELS[key]}")
    print(sep)
    print(f"  Exact Match Rate    : {em:.4f}  ({a['exact']}/{n})")
    print(f"  Precision  (F0.5)   : {p:.4f}")
    print(f"  Recall     (F0.5)   : {r:.4f}")
    print(f"  F0.5                : {f05:.4f}")
    print(f"  GLEU                : {avg_gleu:.4f}")
    print(f"  Oracle F0.5@5       : {o5f05:.4f}  (P={o5p:.4f}  R={o5r:.4f})")
    for k in sorted(a["recall"]):
        print(f"  Recall@{k:<2}            : {a['recall'][k]/n:.4f}  ({a['recall'][k]}/{n})")
    print(f"  Noop Accuracy       : {noop_acc:.4f}  ({a['noop_correct']}/{a['noop_total']})")
    print(f"  TP / FP / FN        : {a['tp']:.1f} / {a['fp']:.1f} / {a['fn']:.1f}")

    pred_key_name = PRED_KEYS[key]
    print(f"\n{'─'*58}")
    print(f"  PER ERROR TYPE  —  {LABELS[key]}")
    print(f"{'─'*58}")
    for etype, stats in sorted(a["type_stats"].items()):
        pe, re_, f05e = compute_f05(stats["tp"], stats["fp"], stats["fn"])
        total_e = sum(1 for it in test_data if it.get("error_type", "UNKNOWN") == etype)
        em_e    = sum(
            1 for it in test_data
            if it.get("error_type", "UNKNOWN") == etype
            and any(normalize(it[pred_key_name]) == normalize(ref)
                    for ref in get_references(it))
        )
        print(f"\n  [{etype}]  ({total_e} samples,  EM={em_e}/{total_e})")
        print(f"    P={pe:.4f}  R={re_:.4f}  F0.5={f05e:.4f}")
        print(f"    TP={stats['tp']:.1f}  FP={stats['fp']:.1f}  FN={stats['fn']:.1f}")

    return {"em": em, "p": p, "r": r, "f05": f05, "gleu": avg_gleu,
            "oracle5_f05": o5f05, "oracle_f05": of05, "noop_acc": noop_acc}

res_nr = print_results_for("nr")
res_rr = print_results_for("rr")

# ======================
# SIDE-BY-SIDE COMPARISON
# ======================

print("\n" + "=" * 60)
print("   SIDE-BY-SIDE COMPARISON")
print("=" * 60)
metrics = [
    ("Exact Match",          "em"),
    ("Precision",            "p"),
    ("Recall",               "r"),
    ("F0.5",                 "f05"),
    ("GLEU",                 "gleu"),
    ("Oracle F0.5@5",        "oracle5_f05"),
    ("Noop Accuracy",        "noop_acc"),
]
print(f"  {'Metric':<24}  {'No Reranker':>11}  {'Reranked':>11}  {'Δ':>8}")
print(f"  {'─'*24}  {'─'*11}  {'─'*11}  {'─'*8}")
for name, k in metrics:
    v1, v2 = res_nr[k], res_rr[k]
    delta  = v2 - v1
    sign   = "+" if delta >= 0 else ""
    print(f"  {name:<24}  {v1:>11.4f}  {v2:>11.4f}  {sign}{delta:.4f}")

# ======================
# SAVE CSV
# ======================

print(f"\nSaving CSV → {OUTPUT_CSV_PATH}")

raw_headers    = [f"raw_beam_{k}"      for k in range(1, NUM_RETURN_SEQ + 1)]
ranked_headers = [f"reranked_cand_{k}" for k in range(1, NUM_RETURN_SEQ + 1)]
header = [
    "#", "source",
    "pred_no_rerank",  "best_ref_nr", "match_nr", "TP_nr", "FP_nr", "FN_nr",
    "pred_reranked",   "best_ref_rr", "match_rr", "TP_rr", "FP_rr", "FN_rr",
    "error_type",
] + raw_headers + ranked_headers

with open(OUTPUT_CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    for idx, item in enumerate(test_data, 1):
        src   = item["incorrect_sentence"]
        refs  = get_references(item)
        etype = item.get("error_type", "UNKNOWN")

        pred_nr = item["pred_no_rerank"]
        pred_rr = item["pred_reranked"]

        match_nr = any(normalize(pred_nr) == normalize(r) for r in refs)
        match_rr = any(normalize(pred_rr) == normalize(r) for r in refs)

        raw_cands    = item.get("raw_candidates",    [])
        ranked_cands = item.get("ranked_candidates", [])
        raw_vals    = (raw_cands    + [""] * NUM_RETURN_SEQ)[:NUM_RETURN_SEQ]
        ranked_vals = (ranked_cands + [""] * NUM_RETURN_SEQ)[:NUM_RETURN_SEQ]

        writer.writerow([
            idx, src,
            pred_nr, item.get("_best_ref_nr", ""), match_nr,
            round(item.get("_tp_nr", 0), 2),
            round(item.get("_fp_nr", 0), 2),
            round(item.get("_fn_nr", 0), 2),
            pred_rr, item.get("_best_ref_rr", ""), match_rr,
            round(item.get("_tp_rr", 0), 2),
            round(item.get("_fp_rr", 0), 2),
            round(item.get("_fn_rr", 0), 2),
            etype,
            *raw_vals,
            *ranked_vals,
        ])

print("✅ CSV saved")

# ======================
# SAVE TXT (human-readable, one sample per block)
# ======================

print(f"\nSaving TXT  → {OUTPUT_TXT_PATH}")

SEP_MAJOR = "=" * 60
SEP_MINOR = "-" * 60

with open(OUTPUT_TXT_PATH, "w", encoding="utf-8") as f:
    for idx, item in enumerate(test_data, 1):
        src   = item["incorrect_sentence"]
        refs  = get_references(item)
        etype = item.get("error_type", "UNKNOWN")

        pred_nr = item["pred_no_rerank"]
        pred_rr = item["pred_reranked"]

        match_nr = any(normalize(pred_nr) == normalize(r) for r in refs)
        match_rr = any(normalize(pred_rr) == normalize(r) for r in refs)

        tp_nr = round(item.get("_tp_nr", 0), 2)
        fp_nr = round(item.get("_fp_nr", 0), 2)
        fn_nr = round(item.get("_fn_nr", 0), 2)
        tp_rr = round(item.get("_tp_rr", 0), 2)
        fp_rr = round(item.get("_fp_rr", 0), 2)
        fn_rr = round(item.get("_fn_rr", 0), 2)

        raw_cands    = item.get("raw_candidates",    [])
        ranked_cands = item.get("ranked_candidates", [])

        f.write(f"{SEP_MAJOR}\n")
        f.write(f"Sample #{idx:<4}  |  {etype}\n")
        f.write(f"{SEP_MAJOR}\n")
        f.write(f"Source        : {src}\n")
        for i, ref in enumerate(refs, 1):
            label = "Reference" if len(refs) == 1 else f"Reference {i}"
            f.write(f"{label:<14}: {ref}\n")

        f.write(f"\n{SEP_MINOR}\n")
        f.write(f"No reranker\n")
        f.write(f"{SEP_MINOR}\n")
        f.write(f"Prediction    : {pred_nr}\n")
        f.write(f"Match         : {match_nr}   TP={tp_nr}  FP={fp_nr}  FN={fn_nr}\n")

        f.write(f"\n{SEP_MINOR}\n")
        f.write(f"Reranked\n")
        f.write(f"{SEP_MINOR}\n")
        f.write(f"Prediction    : {pred_rr}\n")
        f.write(f"Match         : {match_rr}   TP={tp_rr}  FP={fp_rr}  FN={fn_rr}\n")

        f.write(f"\n{SEP_MINOR}\n")
        f.write(f"Raw beam candidates (generator order)\n")
        f.write(f"{SEP_MINOR}\n")
        for i, cand in enumerate(raw_cands, 1):
            f.write(f"  {i}. {cand}\n")

        f.write(f"\n{SEP_MINOR}\n")
        f.write(f"Reranked candidates\n")
        f.write(f"{SEP_MINOR}\n")
        for i, cand in enumerate(ranked_cands, 1):
            f.write(f"  {i}. {cand}\n")

        f.write("\n\n")

print("✅ TXT saved")

# ======================
# SAMPLE PREDICTIONS (sanity check, prints first 5 to console)
# ======================

print("\n===== SAMPLE PREDICTIONS (first 5) =====")
for item in test_data[:5]:
    src  = item["incorrect_sentence"]
    refs = get_references(item)
    nr   = item["pred_no_rerank"]
    rr   = item["pred_reranked"]
    print(f"\nSRC          : {src}")
    print(f"PRED (no rr) : {nr}  match={any(normalize(nr)==normalize(r) for r in refs)}")
    print(f"PRED (rrnk)  : {rr}  match={any(normalize(rr)==normalize(r) for r in refs)}")
    print(f"REF          : {refs[0]}")
    print(f"sys_edits    : {extract_edits(src, nr)}")
    print(f"gold_edits   : {extract_edits(src, refs[0])}")

Loading generator from tuyal/Stage2FFTmBart ...
Generator loaded ✅
Loading reranker (AIsumit123/muril-reranker-mbartF) ...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Reranker loaded ✅  (sep token = '[SEP]')
Loaded 414 test samples ✅
Sample keys: ['incorrect_sentence', 'correct_sentence', 'error_type', 'edit']

Generating predictions...
  10/414 done
  20/414 done
  30/414 done
  40/414 done
  50/414 done
  60/414 done
  70/414 done
  80/414 done
  90/414 done
  100/414 done
  110/414 done
  120/414 done
  130/414 done
  140/414 done
  150/414 done
  160/414 done
  170/414 done
  180/414 done
  190/414 done
  200/414 done
  210/414 done
  220/414 done
  230/414 done
  240/414 done
  250/414 done
  260/414 done
  270/414 done
  280/414 done
  290/414 done
  300/414 done
  310/414 done
  320/414 done
  330/414 done
  340/414 done
  350/414 done
  360/414 done
  370/414 done
  380/414 done
  390/414 done
  400/414 done
  410/414 done
  414/414 done
✅ Generation complete


   RESULTS  —  MT5 top-1  (no reranker)
  Exact Match Rate    : 0.4614  (191/414)
  Precision  (F0.5)   : 0.5896
  Recall     (F0.5)   : 0.4236
  F0.5                : 0.5467
  GLEU  